Implementation of ZINC experiment for the paper 
[Graph Transformers Dream of Electric Flow](https://arxiv.org/abs/2410.16699)
Cheng, Carin, Sra 2025

Graph Transformer implementation is based on
[A generalization of transformer networks to graphs](https://arxiv.org/pdf/2012.09699.pdf)
Dwivedi, Bresson, 2020   


Code for base Graph Transformer is taken from https://github.com/xbresson/CS6208_2023


In [ ]:
import os
import pickle
from utils import Dictionary, MoleculeDataset, MoleculeDGL, Molecule
import dgl
from dgl.data import MiniGCDataset
import dgl.function as fn
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.nn as nn
import time
import networkx as nx
from utils import compute_ncut
import os, datetime
from itertools import chain
import torch.autograd.profiler as profiler
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.set_device(1)

In [ ]:
# Select dataset
dataset_name = 'ZINC'; data_folder_pytorch = 'ZINC_pytorch_clean_20k/'; data_folder_dgl = 'ZINC_dgl_clean_20k/'
#dataset_name = 'QM9'; data_folder_pytorch = '../labs_lecture09/QM9_pytorch/'; data_folder_dgl = '../labs_lecture09/QM9_dgl/'

# Load the number of atom and bond types 
with open(data_folder_pytorch + "atom_dict.pkl" ,"rb") as f: num_atom_type = len(pickle.load(f))
with open(data_folder_pytorch + "bond_dict.pkl" ,"rb") as f: num_bond_type = len(pickle.load(f))
print(num_atom_type)
print(num_bond_type)

# Load the DGL datasets
datasets_zinc = MoleculeDataset(dataset_name, data_folder_dgl)
trainset, valset, testset = datasets_zinc.train, datasets_zinc.val, datasets_zinc.test
print(len(trainset))
print(len(valset))
print(len(testset))
idx = 0
print(trainset[0])
max_nodes = max(max([g[0].num_nodes() for g in trainset]), max([g[0].num_nodes() for g in testset]), max([g[0].num_nodes() for g in valset]),)
max_edges = max(max([g[0].num_edges() for g in trainset]), max([g[0].num_edges() for g in testset]), max([g[0].num_edges() for g in valset]),)
print('max nodes', max_nodes)
print('max edges', max_edges)

def preprocess_B(trainset, max_n=max_nodes, max_e = int(max_edges/2)):
    B_trainset = torch.zeros(len(trainset), max_n, max_e).to(device)
    e_feat_mat = torch.zeros(len(trainset), max_e)
    n_feat_mat = torch.zeros(len(trainset), max_n)
    aug_trainset = [(i, (g[0].to(device), g[1].to(device))) for (i,g) in enumerate(trainset)]
    for t in aug_trainset:
        i = t[0]
        g = t[1][0]
        n_e = g.num_edges()
        n_n = g.num_nodes()
        B, keep_inds, discard_inds = remove_bi(g.inc('both').to_dense().to(device))
        B_trainset[i,0:n_n,0:B.shape[1]] = B#/(B.norm(p=1,dim=1)[:,None]**0.5)
        e_feat_mat[i,0:B.shape[1]] = g.edata['feat'][keep_inds.int()].to(device)
        n_feat_mat[i,0:g.ndata['feat'].shape[0]] = g.ndata['feat'].to(device)
        assert (g.edata['feat'][keep_inds.int()] - g.edata['feat'][discard_inds.int()]).float().norm()==0
    return B_trainset, aug_trainset, e_feat_mat, n_feat_mat

def remove_bi(B):
    indices = (torch.einsum('ij,ik->jk',(B,B)) == -2).nonzero(as_tuple=False)
    indices_list = [tuple(idx.tolist()) for idx in indices]
    indices_list = [(min(a[0],a[1]), max(a[0],a[1])) for a in indices_list]
    sorted_indices = sorted(indices_list)
    #print(sorted_indices)
    keep_inds = torch.tensor(sorted_indices)[0:-1:2][:,0]
    discard_inds = torch.tensor(sorted_indices)[0:-1:2][:,1]
    return B[:,keep_inds], keep_inds, discard_inds

B_trainset, aug_trainset, e_feat_train, n_feat_train = preprocess_B(trainset)
B_testset, aug_testset, e_feat_test, n_feat_test = preprocess_B(testset)
B_valset, aug_valset, e_feat_val, n_feat_val = preprocess_B(valset)



# Add positional encoding feature

In [ ]:
pos_enc_dim = 6 # dimension of PE, ZINCx

# Positional encoding as Laplacian eigenv_feat_feat_feat_feat_feat_feat_feat_feat_feat_feat_feat_featectors
def LapEig_positional_encoding(g, pos_enc_dim):
    Adj = g.adj().to_dense() # Adjacency matrix
    Dn = ( g.in_degrees()** -0.5 ).diag() # Inverse and sqrt of degree matrix
    Lap = torch.eye(g.number_of_nodes()).to(device) - Dn.matmul(Adj).matmul(Dn) # Laplacian operator
    EigVal, EigVec = torch.linalg.eig(Lap) # Compute full EVD
    EigVal, EigVec = EigVal.real, EigVec.real # make eig real
    EigVec = EigVec[:, EigVal.argsort()] # sort in increasing order of eigenvalues
    EigVec = EigVec[:,1:pos_enc_dim+1] # select the first non-trivial "pos_enc_dim" eigenvector
    if EigVec.shape[1] < pos_enc_dim:
        EigVec = torch.cat((EigVec, torch.zeros((EigVec.shape[0], pos_enc_dim-EigVec.shape[1])).to(device)), dim=1)
    return EigVec

# Add node and edge features to graphs
#pos_enc_dim = 3 # dimension of PE, QM9
def add_node_edge_features(dataset):
    for (_, (graph,_)) in dataset:
        graph.ndata['pos_enc'] = LapEig_positional_encoding(graph, pos_enc_dim) # node positional encoding feature 
    return dataset

# Generate graph datasets
aug_trainset = add_node_edge_features(aug_trainset)
aug_testset = add_node_edge_features(aug_testset)
aug_valset = add_node_edge_features(aug_valset)

In [ ]:
max_nodes = max(max([g[1][0].num_nodes() for g in aug_trainset]), max([g[1][0].num_nodes() for g in aug_testset]), max([g[1][0].num_nodes() for g in aug_valset]),)
max_edges = max(max([g[1][0].num_edges() for g in aug_trainset]), max([g[1][0].num_edges() for g in aug_testset]), max([g[1][0].num_edges() for g in aug_valset]),)

print(max_nodes, max_edges, num_atom_type, num_bond_type)

In [ ]:
# collate function prepares a batch of graphs, labels and other graph features (if needed)
def collate(samples):
    # Input sample is a list of pairs (idx, (graph, label))
    (indices, samples) = map(list, zip(*samples))
    graphs, labels = map(list, zip(*samples))
    batch_graphs = dgl.batch(graphs)    # batch of graphs
    batch_labels = torch.stack(labels)  # batch of labels (here chemical target)
    return batch_graphs, batch_labels, indices

In [ ]:
# Define a two-layer MLP for regression 
class MLP_layer(nn.Module): 
    
    def __init__(self, input_dim, hidden_dim): 
        super(MLP_layer, self).__init__()
        self.linear1 = nn.Linear( input_dim, hidden_dim, bias=True )
        self.linear2 = nn.Linear( hidden_dim, 1, bias=True )
        
    def forward(self, x):
        y = self.linear2(torch.relu(self.linear1(x)))
        return y

        
# class graph multi head attention layer  
class graph_MHA_layer(nn.Module): # MHA = Multi Head Attention
    
    def __init__(self, hidden_dim, head_hidden_dim, num_heads): # hidden_dim = d
        super().__init__()
        self.head_hidden_dim = head_hidden_dim # head_hidden_dim = d' = d/K
        self.num_heads = num_heads # number of heads = K
        self.WQ = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True) # define K x W matrix of size=(d',d')
        self.WK = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        self.WV = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        self.WE = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        self.WF = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        self.WG = nn.Linear(hidden_dim, head_hidden_dim * num_heads, bias=True)
        
    # Step 1 of message-passing with DGL: 
    #   Node feature and edge features are passed along edges (src/j => dst/i) 
    def message_func(self, edges): 
        # Compute bi-linear products with edge feature : q_i^T * diag(e_ij) * k_j 
        # You may use "edges.dst[] for i, edges.src[] for j, edges.data[] form ij" 
        qikj = (edges.src['K'] * edges.data['E'] * edges.dst['Q']).sum(dim=2).unsqueeze(2) # size=(E,K,1), edges.src/dst/data[].size=(E,K,d')
        expij = torch.exp( (qikj / torch.sqrt(torch.tensor(self.head_hidden_dim))).clamp(-5, 5) ) # exp_ij = exp( clamp(q_i^T * k_j / sqrt(d')) ), size=(E,K,1)
        vj = edges.src['V'] # size=(E,K,d')
        # Compute edge feature : q_i^T * diag(e_ij) * k_j
        eij = edges.src['K'] * edges.data['E'] * edges.dst['Q'] / torch.sqrt(torch.tensor(self.head_hidden_dim)) # e_ij = q_i^T * diag(E_ij) * k_j / sqrt(d'), size=(E,K,d')
        edges.data['e'] = eij # update edge feature 
        return {'expij' : expij, 'vj' : vj} 
    
    # Step 2 of message-passing with DGL: 
    #   Reduce function collects all messages={hj, eij} sent to node dst/i with Step 1
    #                   and sum/mean over the graph neigbors j in Ni
    def reduce_func(self, nodes):
        expij = nodes.mailbox['expij'] # size=(N,|Nj|,K,1), |Nj|=num_neighbors
        vj = nodes.mailbox['vj'] # size=(N,|Nj|,K,d')
        numerator = torch.sum( expij * vj, dim=1 ) # sum_j exp_ij . v_j, size=(N,K,d')
        denominator = torch.sum( expij, dim=1 ) # sum_j' exp_ij', size=(N,K,1)
        h = numerator / denominator # h_i = sum_j score_ij . v_j , where score_ij = exp_ij / sum_j' exp_ij', size=(N,K,d')
        return {'h' : h} 
    
    def forward(self, g, h, e):
        Q = self.WQ(h) # size=(N, d)
                       # computational trick to compute quickly K linear transformations h_k.WQ of size=(N, d')
                       # first compute linear transformation h.WQ of size=(N, d)
                       # then reshape h.WQ of size=(N, K, d'=d/K)
        K = self.WK(h) # size=(N, d)
        V = self.WV(h) # size=(N, d)
        E = self.WE(e) # size=(E, d)
        F = self.WF(h) # size=(N, d)
        G = self.WG(h) # size=(N, d)
        g.ndata['Q'] = Q.view(-1, self.num_heads, self.head_hidden_dim) # size=(N, K, d'=d/K)
        g.ndata['K'] = K.view(-1, self.num_heads, self.head_hidden_dim) # size=(N, K, d'=d/K)
        g.ndata['V'] = V.view(-1, self.num_heads, self.head_hidden_dim) # size=(N, K, d'=d/K)
        g.edata['E'] = E.view(-1, self.num_heads, self.head_hidden_dim) # size=(E, K, d'=d/K)
        g.ndata['F'] = F.view(-1, self.num_heads, self.head_hidden_dim) # size=(N, K, d'=d/K)
        g.ndata['G'] = G.view(-1, self.num_heads, self.head_hidden_dim) # size=(N, K, d'=d/K)
        g.update_all(self.message_func, self.reduce_func) # compute with DGL the graph MHA 
        gMHA = g.ndata['h'] # size=(N, K, d'=d/K)
        gMHE = g.edata['e'] # size=(E, K, d'=d/K)
        return gMHA, gMHE
    
    
# class GraphTransformer layer  
class GraphTransformer_layer(nn.Module):
    
    def __init__(self, hidden_dim, num_heads, norm, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim # hidden_dim = d
        self.num_heads = num_heads # number of heads = K
        self.dropout = dropout # dropout value
        self.gMHA = graph_MHA_layer(hidden_dim, hidden_dim//num_heads, num_heads) # graph MHA layer
        self.WO = nn.Linear(hidden_dim, hidden_dim) # LL
        self.WOe = nn.Linear(hidden_dim, hidden_dim) # LL
        self.norm = norm # type of layer normalization
        if self.norm == 'LN': # layer normalization
            self.layer_norm1 = nn.LayerNorm(hidden_dim)
            self.layer_norm2 = nn.LayerNorm(hidden_dim)
            self.layer_norm1e = nn.LayerNorm(hidden_dim)
            self.layer_norm2e = nn.LayerNorm(hidden_dim)
        elif self.norm == 'BN': # batch normalization
            self.layer_norm1 = nn.BatchNorm1d(hidden_dim)
            self.layer_norm2 = nn.BatchNorm1d(hidden_dim)
            self.layer_norm1e = nn.BatchNorm1d(hidden_dim)
            self.layer_norm2e = nn.BatchNorm1d(hidden_dim)
        self.linear1 = nn.Linear(hidden_dim, hidden_dim) # LL1 for MLP
        self.linear2 = nn.Linear(hidden_dim, hidden_dim) # LL2 for MLP
        self.linear1e = nn.Linear(hidden_dim, hidden_dim) # LL1 for MLP
        self.linear2e = nn.Linear(hidden_dim, hidden_dim) # LL2 for MLP
        
    def forward(self, g, h, e): 
        
        # Self-attention layer
        h_rc = h # size=(N,d), V=num_nodes, for residual connection
        e_rc = e
        h_MHA, e_MHE = self.gMHA(g, h, e) # MHA, size=(N, K, d'=d/K)
        h_MHA = h_MHA.view(-1, self.hidden_dim) # size=(N, d)
        e_MHE = e_MHE.view(-1, self.hidden_dim) # size=(N, d)
        h_MHA = F.dropout(h_MHA, self.dropout, training=self.training) # dropout, size=(N, d)
        e_MHE = F.dropout(e_MHE, self.dropout, training=self.training) # dropout, size=(N, d)
        h_MHA = self.WO(h_MHA) # LL, size=(N, d)
        e_MHE = self.WOe(e_MHE) # LL, size=(N, d)
        h = h_rc + h_MHA # residual connection, size=(N, d)
        e = e_rc + e_MHE # residual connection, size=(N, d)
        h = self.layer_norm1(h) # layer normalization, size=(N, d)
        e = self.layer_norm1e(e) # layer normalization, size=(N, d)
        
        # Fully-connected layer
        h_rc = h # for residual connection, size=(N, d)
        e_rc = e # for residual connection, size=(N, d)
        h_MLP = self.linear1(h) # LL, size=(H, d)
        e_MLP = self.linear1e(e) # LL, size=(H, d)
        h_MLP = torch.relu(h_MLP) # size=(N, d)
        e_MLP = torch.relu(e_MLP) # size=(N, d)
        h_MLP = F.dropout(h_MLP, self.dropout, training=self.training) # dropout, size=(N, d)
        e_MLP = F.dropout(e_MLP, self.dropout, training=self.training) # dropout, size=(N, d)
        h_MLP = self.linear2(h_MLP) # LL, size=(N, d)
        e_MLP = self.linear2e(e_MLP) # LL, size=(N, d)
        h = h_rc + h_MLP # residual connection, size=(N, d)
        e = e_rc + e_MLP # residual connection, size=(N, d)
        h = self.layer_norm2(h) # layer normalization, size=(N, d)
        e = self.layer_norm2e(e) # layer normalization, size=(N, d)
        
        return h, e
    
    
# class Graph Transformer network
class GraphTransformer_net(nn.Module):
    
    def __init__(self, net_parameters):
        super(GraphTransformer_net, self).__init__()
        input_dim = net_parameters['input_dim']
        pos_enc_dim = net_parameters['pos_enc_dim']
        hidden_dim = net_parameters['hidden_dim']
        num_heads = net_parameters['num_heads']
        norm = net_parameters['norm'] 
        L = net_parameters['L']
        self.embedding_h = nn.Embedding(num_atom_type, hidden_dim)
        self.embedding_e = nn.Embedding(num_bond_type, hidden_dim)
        self.embedding_pe = nn.Linear(pos_enc_dim, hidden_dim)
        self.GraphTransformer_layers = nn.ModuleList([ GraphTransformer_layer(hidden_dim, num_heads, norm) for _ in range(L) ]) 
        self.MLP_layer = MLP_layer(hidden_dim, 1)
        
    def forward(self, g, h, pe, e):
        
        # input node embedding
        h = self.embedding_h(h) # size=(num_nodes, hidden_dim)
        
        # if PE used
        h = h + self.embedding_pe(pe) # size=(num_nodes, hidden_dim)
        
        # input edge embedding
        e = self.embedding_e(e) # size=(num_edges, hidden_dim)
        
        # graph convnet layers
        for GT_layer in self.GraphTransformer_layers:
            h, e = GT_layer(g, h, e) # size=(num_nodes, hidden_dim)
        
        # MLP classifier
        g.ndata['h'] = h
        y = dgl.mean_nodes(g,'h') # DGL mean function over the neighbors, size=(num_graphs, hidden_dim)
        y = self.MLP_layer(y) # size=(num_graphs, num_classes)    
        return y    
    
    def loss(self, y_scores, y_labels):
        loss = nn.L1Loss()(y_scores, y_labels)
        return loss        
        

# Instantiate one network (testing)
net_parameters = {}
net_parameters['input_dim'] = 1
net_parameters['pos_enc_dim'] = pos_enc_dim
net_parameters['hidden_dim'] = 128
net_parameters['num_heads'] = 8
net_parameters['norm'] = 'BN' 
net_parameters['L'] = 4
net = GraphTransformer_net(net_parameters)
print(net)

In [ ]:
def mym(Q, Z):
    if Q.dim()==1:
        assert(False)
        out = Q[:,None,None,None,None]*Z
    elif Q.dim()==2:
        #diagonal case
        out = torch.einsum('Hj, HBMj -> HBMj', (Q,Z))
    else:
        out = torch.einsum('Hij, HBMj -> HBMi', (Q,Z))
    return out

def myproj(Z, v):
    #print(v.shape)
    #print(Z.shape)
    out = torch.einsum('Hi, BMji -> HBMj', (v,Z))
    return out

def attentionZ(B, phi, WP_phi, WQ_phi, WK_phi, bias_phi, WP_B, WQK_B, bias_B, ss_ratio, e_proj, n_proj, my_I):
    P_B = (WP_B[:,None,None,None] * myproj(B, e_proj[:,1,:]))
    
    B = B/( (1e-6+B.norm(p=1,dim=2))[:,:,None,:]**0.5)
    B_QK_B =  WQK_B[:,None,None,None] *  torch.einsum('HBNi, HBMi -> HBNM', (myproj(B, e_proj[:,1,:]), myproj(B, e_proj[:,1,:])))
    
    #P Q K ss for phi
    P_phi = mym(WP_phi, myproj(phi, n_proj[:,1,:]))
    Q_phi = mym(WQ_phi, myproj(phi, n_proj[:,1,:]))
    K_phi = mym(WK_phi, myproj(phi, n_proj[:,1,:]))
    #P_phi = (WP_phi.mean(dim=[1,2])[:,None,None,None] * myproj(phi, n_proj[:,0,:]))
    #Q_phi = (WQ_phi.mean(dim=[1,2])[:,None,None,None] * myproj(phi, n_proj[:,1,:]))
    #K_phi = (WK_phi.mean(dim=[1,2])[:,None,None,None] * myproj(phi, n_proj[:,1,:]))
    phi_QK_phi = torch.einsum('HBNi,HBMi->HBNM',(Q_phi, K_phi))
    
    ss_B = ss_ratio[:,0,0][:,None,None,None] * phi_QK_phi + ss_ratio[:,0,1][:,None,None,None] * B_QK_B + bias_B[:,None,None,None] * my_I[None,None,:,:]
    ss_phi = ss_ratio[:,1,0][:,None,None,None] * phi_QK_phi + ss_ratio[:,1,1][:,None,None,None] * B_QK_B + bias_phi[:,None,None,None] * my_I[None,None,:,:]

    #P times ss
    delta_B = torch.einsum('HBNj, HBNM -> HBMj', (P_B, ss_B))
    delta_phi = torch.einsum('HBNj, HBNM -> HBMj', (P_phi, ss_phi))

    delta_B = delta_B[:,:,:,:,None] * e_proj[:,3,:][:,None,None,None,:]
    delta_phi = delta_phi[:,:,:,:,None] * n_proj[:,3,:][:,None,None,None,:]

    delta_B = delta_B.sum(dim=0)
    delta_phi = delta_phi.sum(dim=0)
    return delta_B, delta_phi


class Transformer_Z(nn.Module):
    def __init__(self, out_phi_dim, n_layer, n_head, n, d, var, num_e_f, num_n_f, normalize=False, phi_dim=4):
        super(Transformer_Z, self).__init__()
        self.register_parameter('WP_phi', torch.nn.Parameter(torch.zeros(n_layer, n_head, phi_dim)))
        self.register_parameter('WQ_phi', torch.nn.Parameter(torch.zeros(n_layer, n_head, phi_dim)))
        self.register_parameter('WK_phi', torch.nn.Parameter(torch.zeros(n_layer, n_head, phi_dim)))
        self.register_parameter('WP_B', torch.nn.Parameter(torch.zeros(n_layer, n_head)))
        self.register_parameter('WQK_B', torch.nn.Parameter(torch.zeros(n_layer, n_head)))

        self.register_parameter('bias_phi', torch.nn.Parameter(torch.zeros(n_layer, n_head)))
        self.register_parameter('bias_B', torch.nn.Parameter(torch.zeros(n_layer, n_head)))

        self.register_parameter('ss_ratio', torch.nn.Parameter(torch.zeros(n_layer, n_head, 2, 2)))

        self.register_parameter('scaling', torch.nn.Parameter(torch.zeros(out_phi_dim, phi_dim, num_n_f)))
        
        self.my_I = torch.eye(n,n).to(device)
        #PQK,expand are 0,1,2,3 resp
        self.register_parameter('e_proj', torch.nn.Parameter(torch.zeros(n_layer, n_head, phi_dim, num_e_f)))
        self.register_parameter('n_proj', torch.nn.Parameter(torch.zeros(n_layer, n_head, phi_dim, num_n_f)))
        self.first_forward=True
            
           # self.scaling.zero_()
        self.n_layer = n_layer
        self.n_head = n_head
        self.n = n
        self.d = d
        self.normalize = normalize
        self.phi_dim = phi_dim
        self.num_n_f = num_n_f

        self.register_parameter('phi_init', torch.nn.Parameter(torch.randn([n,phi_dim, num_n_f])))
        #self.bns = self.linears = nn.ModuleList([MyBatchNorm1d(self.phi_dim) for i in range(3*self.n_layer)])

        with torch.no_grad():
            for p in self.parameters():
                p.normal_(0,var)
            self.WP_phi.zero_()
            self.WP_B.zero_()
            self.bias_phi.zero_()
            self.bias_B.zero_()
            self.WQ_phi.fill_(1)
            self.WK_phi.fill_(1)
            #self.scaling.zero_()
            self.ss_ratio.fill_(1)
            self.e_proj.fill_(1)
            self.n_proj.fill_(1)
            self.phi_init.data = self.phi_init/self.phi_init.norm(p=2, dim=1)[:,None,:]
            self.scaling.data = self.scaling/self.scaling.norm(p=2, dim=1)[:,None]


    #new forward, looping middle layer
    def forward(self, B, phi, mask):
        if self.first_forward:
            print('3x')
            self.first_forward=False
        n = self.n
        d = self.d
        #factor = mask.int().sum(dim=[1,2]) / n
        for i in range(self.n_layer):
            if i%2==1:
                n_loops = 3
            else:
                n_loops = 3
            
            for j in range(n_loops):
                #B = B * mask[:,:,None,:]
                #phi = phi * mask[:,:,None,:]
                
                Bt = B
                phit = phi
                # the forward map of each layer is given by F(Z) = Z + attention(Z)
                
                WP_phi = self.WP_phi[i,:,:]
                WQ_phi = self.WQ_phi[i,:,:]
                WK_phi = self.WK_phi[i,:,:]
                WP_B = self.WP_B[i,:]
                WQK_B = self.WQK_B[i,:]
                bias_phi = self.bias_phi[i,:]
                bias_B = self.bias_B[i,:]
                ss_ratio = self.ss_ratio[i,:,:,:]
    
                e_proj = self.e_proj[i,:,:,:]
                n_proj = self.n_proj[i,:,:,:]
    
                dBt, dphit = attentionZ(B, phi, WP_phi, WQ_phi, WK_phi, bias_phi, WP_B, WQK_B, bias_B, ss_ratio, e_proj, n_proj, self.my_I)
                Bt = Bt + dBt #/ factor[:,None,None,None]
                phit = phit + dphit #/ factor[:,None,None,None]
    
                B = B + Bt
                phi = phi + phit
    
                if self.normalize:
                    B = B / (1e-6 + B.norm(p=2,dim=[1,2])[:,None,None,:].expand(B.shape))
                    phi = phi / (1e-6 + phi.norm(p=2,dim=[1])[:,None,:,:].expand(phi.shape))
                    #phi = self.bns[3*i+j](phi.reshape(phi.shape[0]*phi.shape[1],phi.shape[2]*phi.shape[3])).reshape(phi.shape)
                    #phi = phi - phi.mean(dim=1)[:,None,:]
        #B = B * mask[:,:,None,:]
        #phi = phi * mask[:,:,None,:]
        return B, torch.einsum('ijc,BNjc->BNic', (self.scaling, phi))

# Train the network

In [ ]:
### import numpy as np

n = max_nodes
d = int(max_edges/2)
batch_size = 200
pos_enc_dim = 6
my_pos_enc_dim = 3
pe_norm = 0


def inverse_permutation(perm):
    inv = torch.empty_like(perm)
    inv[perm] = torch.arange(perm.size(0), device=perm.device)
    return inv

def run_one_epoch(net, data_loader, model, B_all, dataset, train=True):
    lst_norm_p = []
    lst_model_norm_p = []
    if train:
        net.train() # during training
        model.train()
    else:
        net.eval()  # during inference/test
        model.eval()
    epoch_loss = 0
    nb_data = 0
    for iter, (batch_graphs, batch_labels, indices) in enumerate(data_loader):
        B = B_all[indices,:,:][:,:,:,None].detach().clone()
        
        idxe = torch.randperm(d)
        B[:,:,:,:]=B[:,:,idxe,:]
        idxn = torch.randperm(n)
        inv_idxn = inverse_permutation(idxn)
        B[:,:,:,:]=B[:,idxn,:,:]

        mask = torch.ones(B.shape[0],B.shape[1],1).to(device)
        null_vids = (B.abs().sum(dim=2) < 1e-6)
        mask[null_vids] = 0
        
        phi = model.phi_init[None,idxn,:,:].expand(B.shape[0],n,phi_dim,num_n_f)
        _, phi = model(B, phi, mask)
        
        predicted_ev = phi[:,inv_idxn,:,0]
        predicted_ev = predicted_ev / (1e-5 + predicted_ev.norm(p=2, dim = 1)[:,None,:])# * 0.3
        #collate variable lengths
        pe_tuple = tuple([predicted_ev[i, 0:dataset[ind][1][0].num_nodes(), :] for (i,ind) in enumerate(indices)])
        batch_pe = torch.cat(pe_tuple, dim=0)
        batch_pe = batch_pe #* 0.3 #* (2 * torch.randint(low=0, high=2, size=(1,pos_enc_dim)).float().to(device) - 1.0 ) #+ batch_graphs.ndata['pos_enc'].to(device)* ( 2 * torch.randint(low=0, high=2, size=(1,pos_enc_dim)).float().to(device) - 1.0 )

        pe_norm = (batch_pe.norm(p=2,dim=0)[None,:].mean()/(len(indices)**0.5)).item()
        
        batch_x = batch_graphs.ndata['feat'].to(device)
        batch_e = batch_graphs.edata['feat'].to(device)

        batch_labels = batch_labels.to(device)
        batch_graphs = batch_graphs.to(device)
        batch_x = batch_x.to(device)
        batch_labels = batch_labels.to(device)
        
        batch_scores = net.forward(batch_graphs.to(device), batch_x, batch_pe, batch_e)
        loss = net.loss(batch_scores, batch_labels)

        if train: # during training, run backpropagation
            optimizer.zero_grad()
            optimizer_model.zero_grad()
            loss.backward()
            
            norm_p, model_norm_p = clip((net.parameters(), model.parameters(), ), clip_r = clip_r)
            lst_norm_p.append(norm_p)
            lst_model_norm_p.append(model_norm_p)
            
            optimizer.step()
            optimizer_model.step()
        epoch_loss += loss.detach().item()
        nb_data += batch_labels.size(0)
        
    if train:
        print('total: [{:.4f}, {:.4f}, {:.4f}, {:.4f}], count={}'
                  .format(np.percentile(lst_norm_p, 25), np.percentile(lst_norm_p, 50), 
                          np.percentile(lst_norm_p, 75), np.percentile(lst_norm_p, 100), 
                          len([num for num in lst_norm_p if num > clip_r])))
        print('model: [{:.4f}, {:.4f}, {:.4f}, {:.4f}]'
                  .format(np.percentile(lst_model_norm_p, 25), np.percentile(lst_model_norm_p, 50), 
                          np.percentile(lst_model_norm_p, 75), np.percentile(lst_model_norm_p, 100)))
    epoch_loss /= (iter + 1)
    return epoch_loss
    


def clip(model_params, clip_r = None):
    norm_p=None
    norm_p = 1e-11
    for p in model_params[0]:
        if p.grad is not None:
            norm_p += p.grad.norm().item()**2
    net_norm_p = norm_p**0.5
    for p in model_params[1]:
        if p.grad is not None:
            norm_p += p.grad.norm().item()**2
    norm_p = norm_p**0.5
    
    model_norm_p = (norm_p**2 - net_norm_p**2)**0.5
    if norm_p > clip_r:
        for p in model_params[0]:
            if p.grad is not None:
                p.grad.mul_(clip_r/norm_p)
        for p in model_params[1]:
            if p.grad is not None:
                p.grad.mul_(clip_r/norm_p)
    return norm_p, model_norm_p
# set net params
net_parameters = {}
net_parameters['input_dim'] = 1
net_parameters['pos_enc_dim'] = pos_enc_dim
net_parameters['hidden_dim'] = 128
net_parameters['num_heads'] = 8
net_parameters['norm'] = 'BN' 
net_parameters['L'] = 4

num_e_f=1
num_n_f=1
phi_dim = 8
clip_r = 10
print('clip_r: {}'.format(clip_r))

# dataset loaders
train_loader = DataLoader(aug_trainset, batch_size=batch_size, shuffle=True, drop_last=True, collate_fn=collate)
test_loader = DataLoader(aug_testset, batch_size=batch_size, shuffle=False, drop_last=False, collate_fn=collate)
val_loader = DataLoader(aug_valset, batch_size=batch_size, shuffle=False, drop_last=False, collate_fn=collate)

run_seeds = [0]#,411,416,419,417] #420
for seed in run_seeds:
    print('seed ', seed)
    hist_list=[]
    print('choosing best initialization')
    best_net = GraphTransformer_net(net_parameters)
    best_model = Transformer_Z(pos_enc_dim, 3, 1, n, d, 0.1, num_e_f = num_e_f, num_n_f = num_n_f, normalize=True, phi_dim=phi_dim).to(device)
    best_net.to(device)
    best_model.to(device)
    best_train_loss = 100
    best_val_loss = 100
    
    best_net_dict = None
    best_model_dict = None
    
    for s in range(3):
        torch.manual_seed(100*seed+s)
        dgl.seed(100*seed+s)
        torch.cuda.manual_seed(100*seed+s)
        
        net = GraphTransformer_net(net_parameters)
        model = Transformer_Z(pos_enc_dim, 3, 1, n, d, 0.1, num_e_f = num_e_f, num_n_f = num_n_f, normalize=True, phi_dim=phi_dim).to(device)
        net_num_p = len(list(net.parameters()))
        model_num_p = len(list(model.parameters()))
        net.to(device)
        model.to(device)
        #model.load_state_dict(torch.load('3x3_nullmask.pth'.format(seed)))
        model.load_state_dict(torch.load('zinc_models/pretrained_{}.pth'.format(seed), map_location=device))
        optimizer = torch.optim.Adam(chain(net.parameters()), lr=0.001)
        optimizer_model = torch.optim.AdamW(chain(model.parameters()), lr=0.001, betas=(0.9,0.99), weight_decay = 0.0001)
    
        #run 1 epoch and check train loss
        start = time.time()
        epoch_train_loss = run_one_epoch(net, train_loader, model, B_trainset, aug_trainset, True)
        epoch_val_loss = run_one_epoch(net, val_loader, model, B_valset, aug_valset, False)  
        print('Seed {}, time {:.4f}, train_loss: {:.4f}, val_loss: {:.4f}, s_norm: {:.4f}'
              .format(s, time.time()-start, epoch_train_loss, epoch_val_loss, model.scaling.norm().item()))
        if (epoch_val_loss < best_val_loss):
            print('improving val loss to {}'.format(epoch_val_loss))
            best_train_loss = epoch_train_loss
            best_val_loss = epoch_val_loss
            best_net.load_state_dict(net.state_dict())
            best_model.load_state_dict(model.state_dict())
    
    print('Using initialization with train = {:.4f} and val = {:.4f}'.format(best_train_loss, best_val_loss))
    
    #load best model
    net.load_state_dict(best_net.state_dict())
    model.load_state_dict(best_model.state_dict())
    optimizer = torch.optim.Adam(chain(net.parameters()), lr=0.001)
    optimizer_model = torch.optim.AdamW(chain(model.parameters()), lr=0.01, betas = (0.9,0.99), weight_decay = 0.0001)
    
    del best_net
    del best_model
    
    
    # initialize net and model
    print('gradual anneal')
    start = time.time()
    for epoch in range(2001):
        optimizer.param_groups[0]['lr'] = 0.001 * 2**(-epoch/800)
        optimizer_model.param_groups[0]['lr'] = 0.01 * 2**(-epoch/800)
        
        epoch_train_loss = run_one_epoch(net, train_loader, model, B_trainset, aug_trainset, True)
        with torch.no_grad(): 
            epoch_test_loss = run_one_epoch(net, test_loader, model, B_testset, aug_testset, False)
            epoch_val_loss = run_one_epoch(net, val_loader, model, B_valset, aug_valset, False)  
        print('Epoch {}, time {:.4f}, train_loss: {:.4f}, test_loss: {:.4f}, val_loss: {:.4f}, s_norm: {:.4f}, pe_norm: {}'.format(epoch, time.time()-start, epoch_train_loss, epoch_test_loss, epoch_val_loss, model.scaling.norm().item(), pe_norm))
        hist_list.append((epoch_train_loss, epoch_test_loss, epoch_val_loss))
        if epoch_val_loss < best_val_loss:
            best_net_dict = net.state_dict().copy()
            best_model_dict = model.state_dict().copy()
            best_val_loss = epoch_val_loss
        if epoch%100==0:
            torch.save((net.state_dict(), model.state_dict()), 'zinc_models/LTGT_{}_epoch_{}.pth'.format(seed, epoch))
            torch.save((best_net_dict,best_model_dict), 'zinc_models/LTGT_{}_best.pth'.format(seed))
    
    torch.save((best_net_dict,best_model_dict,), 'zinc_models/LTGT_{}_best.pth'.format(seed))
    loss_dict = torch.load('zinc_models/loss_dict.pth')
    loss_dict['LTGT_'+str(seed)] = hist_list
    torch.save(loss_dict, 'zinc_models/loss_dict.pth')